In [0]:
%pip install ijson

In [0]:
import json
import ijson
import os
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType
)


In [0]:
file_path = "/Volumes/iics/casemix/history/json_test/sample_500_mb.json"
# file_path="/Volumes/iics/casemix/history/json_test/carereport_hsar_1000183567367217372_20240302_20240321_20240323_003755.json"

target_table = "iics.casemix.json_test_500_mb_variant_chunks"

TARGET_SIZE = 10 * 1024 * 1024

file_name = os.path.basename(file_path)


file_info = dbutils.fs.ls(file_path)[0]

file_size = file_info.size

file_modified = datetime.fromtimestamp(
    file_info.modificationTime / 1000
).date()


LARGE_ARRAY_FIELD = "contained"



with open(file_path, "rb") as f:

    data = json.load(f)


if LARGE_ARRAY_FIELD not in data:

    raise Exception(
        f"Array field '{LARGE_ARRAY_FIELD}' was not found."
    )


entries = data[LARGE_ARRAY_FIELD]

metadata = {
    key: value
    for key, value in data.items()
    if key != LARGE_ARRAY_FIELD
}


print("Number of objects:", len(entries))



rows = []

chunk = []
chunk_size = 0
chunk_no = 0


for entry in entries:

    entry_json = json.dumps(
        entry,
        separators=(",", ":"),
        ensure_ascii=False
    )

    entry_size = len(
        entry_json.encode("utf-8")
    )

    
    if (chunk and chunk_size + entry_size > TARGET_SIZE):

        chunk_no += 1

        chunk_document = metadata.copy()

        chunk_document[LARGE_ARRAY_FIELD] = chunk

        chunk_json = json.dumps(
            chunk_document,
            separators=(",", ":"),
            ensure_ascii=False
        )

        rows.append(
            (
                file_name,
                file_path,
                file_size,
                file_modified,
                chunk_no,
                chunk_json
            )
        )

        chunk = []
        chunk_size = 0




    chunk.append(entry)

    chunk_size += entry_size


if chunk:

    chunk_no += 1

    chunk_document = metadata.copy()

    chunk_document[LARGE_ARRAY_FIELD] = chunk

    chunk_json = json.dumps(
        chunk_document,
        separators=(",", ":"),
        ensure_ascii=False
    )

    rows.append(
        (
            file_name,
            file_path,
            file_size,
            file_modified,
            chunk_no,
            chunk_json
        )
    )


total_chunks = len(rows)

print("Total chunks:", total_chunks)

In [0]:
schema = """
    file_name STRING,
    file_path STRING,
    file_size_bytes BIGINT,
    file_modified DATE,
    chunk_no INT,
    json_text STRING
"""

In [0]:
df = spark.createDataFrame(
    rows,
    schema=schema
)

In [0]:
df = df.withColumn(
    "chunk_size_bytes",
    F.length(
        F.encode(
            F.col("json_text"),
            "UTF-8"
        )
    )
)

In [0]:
variant_df = (
    df
    .select(
        "file_name",
        "file_path",
        "file_size_bytes",
        "file_modified",
        "chunk_no",

        F.lit(total_chunks)
        .cast("int")
        .alias("total_chunks"),

        "chunk_size_bytes",

        F.parse_json(
            F.col("json_text")
        ).alias("json_data")
    )
)

In [0]:
(
    variant_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

In [0]:
%sql
select 
file_name
, file_path
, file_size_bytes
, file_modified
, chunk_no
, total_chunks
, chunk_size_bytes
, json_data
 from iics.casemix.json_test_500_mb_variant_chunks

In [0]:
%sql

SELECT
    file_name,
    file_path,
    file_size_bytes,
    file_modified,
    chunk_no,
    total_chunks,
    chunk_size_bytes,
    length(json_data) AS json_characters
FROM iics.casemix.json_test_500_mb_variant_chunks
ORDER BY chunk_no;

In [0]:
%sql
SELECT
    file_name,
    chunk_no,
    chunk_size_bytes,
    substring(json_data, 1, 1000) AS json_preview
FROM iics.casemix.json_test_500_mb_variant_chunks
ORDER BY chunk_no
LIMIT 10;

In [0]:
%sql
select 
file_name
, file_path
, file_size_bytes
, file_modified
, chunk_no
, total_chunks
, chunk_size_bytes
, json_data
 from iics.casemix.json_test_variant_chunks

In [0]:
%sql
DESCRIBE iics.casemix.json_test_variant_chunks

In [0]:
%sql
SELECT
    chunk_no,
    json_data:id
FROM iics.casemix.json_test_variant_chunks